# Task 3 — Parte A (ablação) e Parte B (regressão logística e árvore de decisão)

Este notebook cobre a **ablação por grupos de variáveis** e os **dois modelos individuais** desta parte do trabalho: **regressão logística** (linear) e **árvore de decisão**.

**Como funciona**

- Usa o `avaliar_folds` e a restante infraestrutura da Task 2 (`avaliacao.py`): 4 folds temporais, orçamento de 200 alertas por 1000 sessões, métrica principal = **recall@200**. O teste (out–dez/2021) é removido ao carregar os dados e nunca é usado aqui.
- O pré-processamento está em `ablacao.py` e segue as regras da Task 1 (imputação pela mediana, limite no percentil 99, log1p, escalonamento, one-hot). É **reajustado em cada fold**, só com o treino desse fold.
- Os hiperparâmetros são **fixos e iguais** em todas as combinações da ablação, para que as diferenças venham da informação e não do ajuste. A afinação vem depois (secção 7).

**Decisões que podem mudar (estão no bloco de configuração):** nível espacial do contexto (`ESPACO`) e hiperparâmetros. Se a colega decidir outra coisa, basta alterar esse bloco e voltar a correr.

| Secção | O que responde |
|---|---|
| 1 | Referência a superar (baselines) |
| 2 | Teste de sanidade: o circuito funciona? |
| 3 | Que nível espacial usar (posto, localização ou distrito)? |
| 4 | Ablação principal: 4 combinações × 2 modelos × 4 folds |
| 5 | A tarifa acrescenta algo além da hora? |
| 6 | Análises de robustez prometidas na secção 8.10 do relatório |
| 7 | Afinação dos hiperparâmetros (por fazer) |
| 8 | Perguntas do enunciado: onde escrever as respostas |

## 0. Configuração e dados

In [5]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd

import ablacao as ab
from avaliacao import (avaliar, avaliar_folds, agregar_folds, comparar,
                       alertas_por_orcamento, folds_temporais, FOLDS)

warnings.filterwarnings("ignore", category=FutureWarning)   # os avisos de convergência continuam visíveis
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

DATA_PATH = Path("Charging_Data_educational.csv")

X_ALERTAS = 200            # orçamento de alertas por 1000 sessões (decidido na Task 2)
ESPACO = "posto"           # nível espacial no contexto: "posto" | "localizacao" | "distrito"
MODELOS = ["logistica", "arvore"]
print("Hiperparâmetros fixos:", ab.HIPERPARAMETROS)

Hiperparâmetros fixos: {'logistica': {'C': 0.1, 'max_iter': 300}, 'arvore': {'max_depth': 8, 'min_samples_leaf': 200}}


In [6]:
df = ab.carregar_dev(DATA_PATH)            # só treino e validação (dez/2019 – set/2021)
print(f"{len(df):,} sessões | {df['Start Time'].min():%Y-%m-%d} a {df['Start Time'].max():%Y-%m-%d} | prevalência {df['is_Abnormal'].mean():.1%}")
assert df["Start Time"].max() < pd.Timestamp("2021-10-01"), "o teste não pode estar carregado"

360,996 sessões | 2019-12-31 a 2021-09-30 | prevalência 18.8%


## 1. Referência a superar

A Task 2 fixou, como referência, a regra **«a última sessão concluída do posto foi anómala»** (recall@200 médio de 0,511) e a taxa de anomalias do posto a 7 dias. Recalculam-se aqui com os mesmos folds, para confirmar que o circuito reproduz os valores da Task 2.

In [7]:
def metricas_fold(res):
    return pd.Series({"recall@200 (média)": res["recall"].mean(), "desvio": res["recall"].std(ddof=1),
                      "pior fold": res["recall"].min(),
                      "PR-AUC norm. (média)": res["pr_auc_norm"].mean(), "ROC-AUC (média)": res["roc_auc"].mean()})

baselines = {
    "regra: última sessão do posto anómala": avaliar_folds(df, lambda tr, va: va["post_last_completed_abnormal"].to_numpy(), X_ALERTAS),
    "taxa de anomalias do posto (7 dias)":   avaliar_folds(df, lambda tr, va: va["post_previous_abnormal_rate_7D"].to_numpy(), X_ALERTAS),
}
ref = pd.DataFrame({k: metricas_fold(v) for k, v in baselines.items()}).T.round(3)
ref

,recall@200 (média),desvio,pior fold,PR-AUC norm. (média),ROC-AUC (média)
regra: última sessão do posto anómala,0.511,0.020,0.482,0.201,0.698
taxa de anomalias do posto (7 dias),0.499,0.024,0.470,0.404,0.772


## 2. Teste de sanidade

Antes das 8 corridas da ablação, corre-se **cada modelo, com todas as variáveis** (combinação 4), nos 4 folds. O que se espera:

- Melhor do que a referência, mas **sem saltos enormes**: o sinal vem sobretudo dos históricos, que a regra da última sessão já capta.
- Um resultado muito acima (por exemplo, recall perto de 0,9) seria sinal de **fuga de informação**: verificar se entrou alguma variável pós-sessão ou se algo foi ajustado com dados de validação.

In [8]:
GRUPOS_TODOS = ab.CONFIGURACOES["4 + histórico do utilizador"]
sanidade = {}
for m in MODELOS:
    sanidade[m] = avaliar_folds(df, ab.fabricar_scorer(m, GRUPOS_TODOS, ESPACO), X_ALERTAS, devolver_scores=True)

linhas = {m: metricas_fold(r[0]) for m, r in sanidade.items()}
linhas["(referência) última sessão do posto"] = metricas_fold(baselines["regra: última sessão do posto anómala"])
pd.DataFrame(linhas).T.round(3)

,recall@200 (média),desvio,pior fold,PR-AUC norm. (média),ROC-AUC (média)
logistica,0.580,0.015,0.566,0.521,0.831
arvore,0.575,0.012,0.563,0.522,0.827
(referência) última sessão do posto,0.511,0.020,0.482,0.201,0.698


In [9]:
# recall@200 por fold: modelos vs referência
ref_regra = baselines["regra: última sessão do posto anómala"]["recall"]
por_fold = pd.DataFrame({**{m: r[0]["recall"] for m, r in sanidade.items()}, "referência": ref_regra}).round(3)
por_fold["logística − referência"] = (por_fold["logistica"] - por_fold["referência"]).round(3)
por_fold["árvore − referência"] = (por_fold["arvore"] - por_fold["referência"]).round(3)
por_fold

,logistica,arvore,referência,logística − referência,árvore − referência
F1,0.590,0.586,0.520,0.070,0.066
F2,0.568,0.563,0.482,0.086,0.081
F3,0.594,0.585,0.526,0.068,0.059
F4,0.566,0.568,0.518,0.048,0.050


## 3. Nível espacial no contexto

A Task 1 mostrou que posto, localização e distrito são redundantes entre si (cada posto determina os outros dois), e a secção 8.2 do relatório prevê compará-los como alternativas. Compara-se com a **árvore** (rápida) e com todas as variáveis, incluindo também a opção **sem informação espacial**. Escolhe-se pela regra da secção 8.9 (maior recall médio, melhor em pelo menos 3 folds, pior fold não inferior).

In [10]:
espaciais = {}
for esp in ["posto", "localizacao", "distrito", None]:
    espaciais[esp or "sem espaço"] = avaliar_folds(df, ab.fabricar_scorer("arvore", GRUPOS_TODOS, esp), X_ALERTAS)

tab_esp = pd.DataFrame({k: metricas_fold(v) for k, v in espaciais.items()}).T.round(3)
tab_esp

,recall@200 (média),desvio,pior fold,PR-AUC norm. (média),ROC-AUC (média)
posto,0.575,0.012,0.563,0.522,0.827
localizacao,0.576,0.011,0.564,0.522,0.826
distrito,0.577,0.013,0.562,0.522,0.827
sem espaço,0.576,0.013,0.563,0.522,0.827


In [11]:
# comparação emparelhada com o posto (A = alternativa, B = posto)
base = espaciais["posto"]
pd.DataFrame({k: comparar(v, base, "recall") for k, v in espaciais.items() if k != "posto"}).T.round(3)

,diferença média,desvio da diferença,folds em que A ganha,pior fold A,pior fold B
localizacao,0.000,0.001,1.0,0.564,0.563
distrito,0.001,0.002,2.0,0.562,0.563
sem espaço,0.001,0.002,1.0,0.563,0.563


## 4. Ablação principal

Quatro combinações **acumuladas**, avaliadas com os dois modelos, nos 4 folds:

1. contexto (tempo + nível espacial) + meteorologia
2. \+ tarifa
3. \+ histórico do posto
4. \+ histórico do utilizador

Na regressão logística, a hora e o dia da semana entram em **one-hot**, para o modelo conseguir reproduzir qualquer padrão horário. Só assim a pergunta «a tarifa acrescenta algo além da hora?» tem resposta honesta (o preço é função da hora). Na árvore, entram como números.

In [12]:
ablacao = {}     # (configuração, modelo) -> (resultados por fold, scores)
for nome_cfg, grupos in ab.CONFIGURACOES.items():
    for m in MODELOS:
        ablacao[(nome_cfg, m)] = avaliar_folds(df, ab.fabricar_scorer(m, grupos, ESPACO), X_ALERTAS, devolver_scores=True)

linhas = {}
for (cfg, m), (res, _) in ablacao.items():
    linhas[(m, cfg)] = metricas_fold(res)
tab_abl = pd.DataFrame(linhas).T.round(3)
tab_abl

,,recall@200 (média),desvio,pior fold,PR-AUC norm. (média),ROC-AUC (média)
logistica,1 contexto + meteorologia,0.375,0.018,0.354,0.177,0.710
arvore,1 contexto + meteorologia,0.275,0.031,0.232,0.054,0.551
logistica,2 + tarifa,0.375,0.018,0.354,0.177,0.710
arvore,2 + tarifa,0.275,0.030,0.234,0.054,0.552
logistica,3 + histórico do posto,0.550,0.015,0.529,0.491,0.804
arvore,3 + histórico do posto,0.557,0.017,0.534,0.491,0.807
logistica,4 + histórico do utilizador,0.580,0.015,0.566,0.521,0.831
arvore,4 + histórico do utilizador,0.575,0.012,0.563,0.522,0.827


In [13]:
# recall@200 por fold, para ver a consistência no tempo
por_fold = pd.DataFrame({(m, cfg): res["recall"] for (cfg, m), (res, _) in ablacao.items()}).round(3)
por_fold.columns = pd.MultiIndex.from_tuples(por_fold.columns, names=["modelo", "configuração"])
por_fold.T

,,F1,F2,F3,F4
modelo,configuração,,,,
logistica,1 contexto + meteorologia,0.354,0.398,0.376,0.373
arvore,1 contexto + meteorologia,0.232,0.278,0.287,0.303
logistica,2 + tarifa,0.354,0.398,0.379,0.370
arvore,2 + tarifa,0.234,0.278,0.287,0.303
logistica,3 + histórico do posto,0.553,0.529,0.563,0.553
arvore,3 + histórico do posto,0.560,0.534,0.574,0.561
logistica,4 + histórico do utilizador,0.590,0.568,0.594,0.566
arvore,4 + histórico do utilizador,0.586,0.563,0.585,0.568


In [14]:
# efeito de cada grupo acrescentado: configuração seguinte vs anterior (diferenças emparelhadas, fold a fold)
nomes = list(ab.CONFIGURACOES)
linhas = {}
for m in MODELOS:
    for ant, seg in zip(nomes[:-1], nomes[1:]):
        c = comparar(ablacao[(seg, m)][0], ablacao[(ant, m)][0], "recall")
        linhas[(m, f"{seg}  vs  {ant}")] = c
efeitos = pd.DataFrame(linhas).T
efeitos[["diferença média", "desvio da diferença"]] = efeitos[["diferença média", "desvio da diferença"]].astype(float).round(4)
efeitos[["diferença média", "desvio da diferença", "folds em que A ganha"]]

diferença média  desvio da diferença  folds em que A ganha
logistica 2 + tarifa  vs  1 contexto + meteorologia                    0.0001               0.0020                   1.0
          3 + histórico do posto  vs  2 + tarifa                       0.1742               0.0298                   4.0
          4 + histórico do utilizador  vs  3 + histórico ...           0.0300               0.0120                   4.0
arvore    2 + tarifa  vs  1 contexto + meteorologia                    0.0005               0.0010                   1.0
          3 + histórico do posto  vs  2 + tarifa                       0.2817               0.0333                   4.0
          4 + histórico do utilizador  vs  3 + histórico ...           0.0182               0.0105                   4.0